# BankIntent AI

> **Tagline:** DistilBERT-powered banking intent classification for 77 customer-support intents.

A self-contained experimentation notebook for fine-tuning **DistilBERT** on the **Banking77** dataset. This notebook executes the complete end-to-end pipeline (training, evaluation, curve plotting, and inference).

### 1. Setup & Configuration

In [ ]:
import os
import json
from pathlib import Path
import torch
import wandb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from dotenv import load_dotenv

load_dotenv()

# ── PATH RESOLUTION ────────────────────
# Resolves project root whether running from repo root or notebooks/ folder
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = str(BASE_DIR / "best_model")
OUTPUT_DIR = str(BASE_DIR / "outputs")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── CONFIG ─────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 3
LR         = 2e-5
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root:     {BASE_DIR}")
print(f"Model save dir:   {MODEL_DIR}")
print(f"Outputs save dir: {OUTPUT_DIR}")
print(f"Using device:     {DEVICE}")

### 2. Experiment Tracking (Weights & Biases)

In [ ]:
wandb_mode = "online" if os.getenv("WANDB_API_KEY") else "disabled"
wandb.init(
    project="bankintent-ai",
    mode=wandb_mode,
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_len": MAX_LEN
    }
)

### 3. Load Banking77 Dataset & Tokenizer

In [ ]:
print("Loading Banking77 dataset (mteb/banking77)...")
dataset    = load_dataset("mteb/banking77")
train_data = dataset["train"]
test_data  = dataset["test"]

num_labels  = 77
id2label    = dict(sorted({x["label"]: x["label_text"] for x in train_data}.items()))
label2id    = {name: i for i, name in id2label.items()}
label_names = list(id2label.values())

print(f"Train size: {len(train_data)} | Test size: {len(test_data)}")
print(f"Sample Query:  \"{train_data[0]['text']}\"")
print(f"Sample Intent: {id2label[train_data[0]['label']]}")

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

### 4. Custom Dataset & DataLoaders

In [ ]:
class BankingDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text  = self.data[idx]["text"]
        label = self.data[idx]["label"]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label":          torch.tensor(label, dtype=torch.long)
        }

train_dataset = BankingDataset(train_data, tokenizer, MAX_LEN)
test_dataset  = BankingDataset(test_data, tokenizer, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 5. Model, Optimizer & Scheduler Setup

In [ ]:
print("Loading DistilBERT model...")
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

### 6. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="weighted")
    return avg_loss, acc, f1

def evaluate(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["label"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="weighted")
    return avg_loss, acc, f1, all_preds, all_labels

### 7. Run Training Loop & Save Best Model (`best_model/`)

In [ ]:
print("Starting training...")
best_f1 = 0.0

# Record epoch history for visualization
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1": [],
    "val_f1": []
}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, test_loader)

    print(f"  Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": train_loss,
        "train_acc":  train_acc,
        "train_f1":   train_f1,
        "val_loss":   val_loss,
        "val_acc":    val_acc,
        "val_f1":     val_f1
    })

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print(f"  ✅ Saved best model to: {MODEL_DIR} (Val F1: {best_f1:.4f})")

print(f"\n🎉 Training complete! Best Validation F1: {best_f1:.4f}")

### 8. Plot & Save Training Curves (`outputs/training_curves.png`)

In [ ]:
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o", color="#2563eb", linewidth=2)
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s", color="#dc2626", linewidth=2)
plt.title("Training & Validation Loss", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

# F1 plot
plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["train_f1"], label="Train F1", marker="o", color="#16a34a", linewidth=2)
plt.plot(history["epoch"], history["val_f1"], label="Val F1", marker="s", color="#9333ea", linewidth=2)
plt.title("Training & Validation Weighted F1", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(curves_path, dpi=300)
plt.show()

print(f"✅ Training curves saved to: {curves_path}")

### 9. Evaluate Best Model & Save Output Details (`outputs/`)
Loads `best_model/` and writes:
- `outputs/metrics.json`
- `outputs/classification_report.txt`
- `outputs/confusion_matrix.png`

In [ ]:
print(f"Evaluating best saved model from: {MODEL_DIR}...")
eval_model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
eval_tokenizer = DistilBertTokenizer.from_pretrained(MODEL_DIR)

test_loss, test_acc, test_f1, final_preds, final_labels = evaluate(eval_model, test_loader)

print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test F1:       {test_f1 * 100:.2f}%")

# 1. Save outputs/metrics.json
metrics_file = os.path.join(OUTPUT_DIR, "metrics.json")
metrics_data = {
    "accuracy": float(test_acc),
    "f1_weighted": float(test_f1),
    "samples": len(final_labels)
}
with open(metrics_file, "w") as f:
    json.dump(metrics_data, f, indent=4)
print(f"✅ Saved metrics: {metrics_file}")

# 2. Save outputs/classification_report.txt
report_file = os.path.join(OUTPUT_DIR, "classification_report.txt")
report_text = classification_report(final_labels, final_preds, target_names=label_names, zero_division=0)
with open(report_file, "w") as f:
    f.write(report_text)
print(f"✅ Saved classification report: {report_file}")

# 3. Save outputs/confusion_matrix.png
cm = confusion_matrix(final_labels, final_preds)
plt.figure(figsize=(20, 20))
sns.heatmap(cm, cmap="Blues")
plt.title("Banking77 Confusion Matrix", fontsize=14, fontweight="bold")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.show()
print(f"✅ Saved confusion matrix: {cm_path}")

wandb.finish()

### 10. Test Sample Predictions

In [ ]:
queries = [
    "Why was my card payment declined?",
    "I want to exchange USD to EUR",
    "How do I activate my new card?",
    "I lost my card, please block it immediately"
]

eval_model.eval()
print("=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

for query in queries:
    inputs = eval_tokenizer(
        query,
        return_tensors="pt",
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        logits = eval_model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_id].item()

    print(f"\nQuery:            \"{query}\"")
    print(f"Predicted Intent: {id2label[pred_id]}")
    print(f"Confidence:       {confidence * 100:.2f}%")

### 11. Publish Model to Hugging Face Hub
Upload the trained model and tokenizer from `best_model/` to the Hugging Face Model Hub using `HF_TOKEN`.

In [ ]:
from huggingface_hub import HfApi

# Read credentials from .env or specify manually
hf_token   = os.getenv("HF_TOKEN")
hf_repo_id = os.getenv("HF_REPO_ID", "Parthiv-Jethva/bankintent-ai")

if not hf_token or hf_token == "your_huggingface_token_here":
    print("⚠️ HF_TOKEN not set or using placeholder in .env. Skipping upload.")
else:
    print(f"Uploading {MODEL_DIR} to Hugging Face Hub repo: '{hf_repo_id}'...")
    api = HfApi(token=hf_token)
    
    # Create repo on Hugging Face (if it doesn't already exist)
    api.create_repo(repo_id=hf_repo_id, exist_ok=True, repo_type="model")
    
    # Upload the entire best_model directory (model weights, config, tokenizer)
    api.upload_folder(
        folder_path=MODEL_DIR,
        repo_id=hf_repo_id,
        repo_type="model"
    )
    
    print(f"🎉 Model successfully published!")
    print(f"🔗 View on Hugging Face: https://huggingface.co/{hf_repo_id}")